# Prepare existing resources for the fixed multi-view evaluation

This reusable entry point executes one configured preparation operation. The launch kit binds existing Kaggle datasets and upstream outputs automatically; it does not re-upload the raw dataset. Models and the two selected scoring methods stay fixed. All scientific output, private checkpoints and reports remain separate.

Operations are explicit: method freeze, private RGB index, model-only RGB release, neutral reference geometry/correspondence, missing-feature and cache shards, cache merge, and post-prediction truth conversion. Only truth conversion opens anomaly outcomes, after complete model predictions verify. The trusted raw resource preparer may read mixed source metadata but exports only whitelisted neutral fields.

In [ ]:
from pathlib import Path
import os
import json
RUNTIME_PROFILE = None
NOTEBOOK_RELATIVE = 'notebooks/input-preparation/current-kaggle-preparation.ipynb'
try:
    from pathlib import Path, PurePosixPath
    import hashlib
    import importlib.util
    import json
    import os
    import shutil
    import stat
    import sys
    import tempfile
    import zipfile

    # Bounded input lookup supports flat, namespaced and automatically expanded mounts.
    def _children(path):
        if not path.is_dir():
            return []
        children = sorted(p for p in path.iterdir() if p.is_dir())
        if len(children) > 250:
            raise ValueError('Source discovery bound exceeded; set VPI_CODE_ROOT explicitly')
        return children

    def _source_candidates():
        explicit = os.getenv('VPI_CODE_ROOT')
        if explicit:
            return [Path(explicit).expanduser()]
        mount = Path('/kaggle/input')
        roots = [Path.cwd(), *Path.cwd().parents]
        roots += [p for p in _children(mount) if p.name not in {'datasets', 'notebooks', 'models'}]
        for category, depth in (('datasets', 2), ('notebooks', 2), ('models', 4)):
            current = [mount / category]
            for _ in range(depth):
                current = [child for parent in current for child in _children(parent)]
                if len(current) > 1000:
                    raise ValueError('Source mount inventory exceeded its bound')
            for parent in current:
                roots += _children(parent / 'versions') if (parent / 'versions').is_dir() else [parent]
        return [root / suffix for root in roots for suffix in ('', 'code', 'repository-files', 'source')]

    def _archive_source():
        configured = os.getenv('VPI_SOURCE_ARCHIVE')
        if not configured:
            raise FileNotFoundError('Mount the prepared source package or set VPI_CODE_ROOT explicitly')
        archive = Path(configured)
        expected = os.getenv('VPI_SOURCE_ARCHIVE_SHA256')
        if not expected or archive.stat().st_size > 256 * 1024 * 1024 or hashlib.sha256(archive.read_bytes()).hexdigest() != expected:
            raise ValueError('Explicit source archive checksum or size check failed')
        destination = Path(os.getenv('VPI_SOURCE_DEST', '/kaggle/working/current-source')).resolve()
        if Path('/kaggle').is_dir() and not destination.is_relative_to(Path('/kaggle/working')):
            raise ValueError('Extracted source must be under /kaggle/working')
        if destination.exists():
            raise ValueError('Source destination exists; verify it with VPI_CODE_ROOT instead of overwriting')
        destination.parent.mkdir(parents=True, exist_ok=True)
        temporary = Path(tempfile.mkdtemp(prefix='current-source.partial-', dir=destination.parent))
        try:
            with zipfile.ZipFile(archive) as bundle:
                infos = bundle.infolist()
                names = [i.filename for i in infos]
                if len(names) != len(set(names)) or sum(i.file_size for i in infos) > 256 * 1024 * 1024 or bundle.testzip():
                    raise ValueError('Source archive inventory, size or CRC is invalid')
                for info in infos:
                    name = info.filename
                    path = PurePosixPath(name)
                    if info.is_dir() or path.is_absolute() or '\\' in name or any(p in {'', '.', '..'} for p in name.split('/')) or stat.S_ISLNK(info.external_attr >> 16):
                        raise ValueError('Unsafe source archive member')
                    output = temporary / name
                    output.parent.mkdir(parents=True, exist_ok=True)
                    output.write_bytes(bundle.read(info))
            os.replace(temporary, destination)
        except BaseException:
            shutil.rmtree(temporary)
            raise
        return destination

    _matches = sorted({p.resolve() for p in _source_candidates() if (p / 'prepared_source_manifest.json').is_file()})
    if len(_matches) > 1:
        raise ValueError('Multiple prepared source roots found; set VPI_CODE_ROOT explicitly')
    CODE_ROOT = _matches[0] if _matches else _archive_source()
    _manifest = json.loads((CODE_ROOT / 'prepared_source_manifest.json').read_text())
    # Verify the bootstrap itself before executing any code from the mounted package.
    _helper = CODE_ROOT / 'scripts/current_notebook_runtime.py'
    if _helper.is_symlink() or hashlib.sha256(_helper.read_bytes()).hexdigest() != _manifest.get('files', {}).get('scripts/current_notebook_runtime.py'):
        raise ValueError('Prepared bootstrap source checksum differs')
    _spec = importlib.util.spec_from_file_location('current_notebook_runtime', _helper)
    _runtime = importlib.util.module_from_spec(_spec)
    _spec.loader.exec_module(_runtime)
    BOOTSTRAP = _runtime.bootstrap(CODE_ROOT, NOTEBOOK_RELATIVE, runtime_profile=RUNTIME_PROFILE)
    CODE_PROVENANCE = BOOTSTRAP['provenance']
    ROOT = CODE_ROOT
    # Scientific imports are allowed only after the exact runtime/source preflight.
    from IPython.display import display, Image
    import pandas as pd
    display({'source': CODE_PROVENANCE, 'runtime': BOOTSTRAP['runtime'],
             'preparation_only': 'Verified source/runtime does not establish protected-resource readiness'})
except BaseException as _bootstrap_error:
    _failure_root = Path('/kaggle/working/runtime-receipts') if Path('/kaggle').is_dir() else Path.cwd()/'.local/runtime-receipts'
    _failure_root.mkdir(parents=True, exist_ok=True)
    _failure = {'status':'bootstrap_failed','error_type':type(_bootstrap_error).__name__,'scientific_run_started':False}
    _temporary = _failure_root/'bootstrap-failure.json.partial'
    _temporary.write_text(json.dumps(_failure,sort_keys=True)+'\n')
    os.replace(_temporary,_failure_root/'bootstrap-failure.json')
    raise

In [ ]:
CONFIG = _runtime.resolve_config('preparation', ROOT)
CONFIG['output_root'] = str(_runtime.writable_output(CONFIG['output_root']))
from current_stage_execution import run_stage
ACTION = CONFIG['action']

In [ ]:
def prepare():
    if ACTION == 'freeze':
        from vpi_next_evaluation.contracts import prepare_method_freeze
        return prepare_method_freeze(CONFIG['reference_root'], CONFIG['output_root'], ROOT)
    if ACTION in {'rgb_index','rgb_release'}:
        from vpi_next_evaluation.native_rgb import prepare_rgb_index, release_rgb
        if ACTION == 'rgb_release' and CONFIG.get('role') == 'unseen':
            from current_rgb_transport import reuse_completed_transport
            retained = reuse_completed_transport(CONFIG['output_root'])
            if retained is not None:
                return retained
        return (prepare_rgb_index if ACTION=='rgb_index' else release_rgb)(CONFIG,ROOT)
    if ACTION == 'raw_geometry':
        from current_official_geometry import prepare_official_mask_geometry
        return prepare_official_mask_geometry(CONFIG)
    if ACTION == 'raw_truth':
        from current_truth_conversion import convert_raw_evaluator_with_typed_rejections
        return convert_raw_evaluator_with_typed_rejections(CONFIG)
    if ACTION == 'neutral':
        from vpi_next_evaluation.native_evaluator import prepare_neutral
        return prepare_neutral(CONFIG)
    if ACTION == 'cache_merge':
        from vpi_next_evaluation.cache_shards import merge_cache_shards
        return merge_cache_shards(CONFIG,ROOT)
    if ACTION == 'region_merge':
        from vpi_next_evaluation.region_shards import merge_region_shards
        return merge_region_shards(CONFIG,ROOT)
    if ACTION == 'prepare_shard':
        from vpi_next_evaluation.native_cache import prepare_shard
        return prepare_shard(CONFIG,ROOT)
    if ACTION == 'official_exposure':
        from current_official_exposure import prepare_official_exposure_from_geometry
        return prepare_official_exposure_from_geometry(CONFIG)
    raise ValueError('Unknown prepared launch operation')

RESULT = run_stage('preparation-'+ACTION,CONFIG,prepare,CODE_PROVENANCE)
if ACTION == 'rgb_release' and CONFIG.get('role') == 'unseen':
    from current_rgb_transport import compact_completed_rgb
    RGB_TRANSPORT = compact_completed_rgb(CONFIG['output_root'],RESULT,role='unseen')
    display({'rgb_transport_status':RGB_TRANSPORT['status'],
             'archive_sha256':RGB_TRANSPORT['archive_sha256'],
             'scene_count':RGB_TRANSPORT['scene_count'],
             'model_file_count':RGB_TRANSPORT['file_count']})
display(RESULT)

## Completion and recovery

A checkpointed result is an incomplete job. Keep its retained output and resume with the same configuration; never let a downstream job consume a partial cohort. Cache merging verifies exact membership and rejects overlapping or missing shards. The final decision still uses the complete declared cohort denominator.

Download each completed job once and verify all registered hashes. Keep source/checkpoints and private evaluator resources outside the later GitHub overlay. Tracking and figure repair use saved outputs without another model run.